### 1.1 : Préparer les bibliothèques et les chemins

L'objectif est d'explorer le dataset image afin de récupérer les caractéristiques
principales de chaque fichier.

Pour chaque image, nous allons relever :

- son nom ;
- sa classe ;
- son format ;
- son mode ;
- sa largeur et sa hauteur ;
- l'écart-type de ses pixels ;
- son nombre de canaux ;
- sa taille en octets.

Les fichiers corrompus seront également pris en compte afin qu'ils ne bloquent
pas l'exploration du dataset.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError


RAW_DIR = Path("../data/raw")

CLASSES = [
    "cardboard",
    "glass",
    "metal",
    "paper",
    "plastic",
    "trash",
]

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".gif",
    ".webp",
    ".tif",
    ".tiff",
}

In [ ]:
#verif 

print(f"Dossier du dataset : {RAW_DIR}")
print(f"Classes : {CLASSES}")

### 1.2 : Fonction d'analyse d'une image

La fonction `analyser_image()` inspecte une image sans la modifier.

Elle récupère ses métadonnées et calcule l'écart-type des pixels.

Si l'image est corrompue ou illisible, la fonction retourne tout de même
les informations disponibles et indique que l'image est corrompue.

In [17]:
def analyser_image(image_path, classe):
    """Analyse une image et retourne ses caractéristiques."""
    
    taille_octets = image_path.stat().st_size

    resultat = {
        "nom": image_path.name,
        "classe": classe,
        "format": None,
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "ecart_type_pixels": None,
        "nombre_canaux": None,
        "taille_octets": taille_octets,
        "corrompue": False,
    }

    try:
        # premiere ouverture pour verifier l'integrite du fichier
        with Image.open(image_path) as image:

            # verifier si l'image est exploitable
            image.verify()

        # deuxieme ouverture pour acceder reellement aux pixels
        with Image.open(image_path) as image:
            resultat["format"] = image.format
            resultat["mode"] = image.mode
            resultat["largeur"], resultat["hauteur"] = image.size

            pixels = np.asarray(image)

            if pixels.ndim == 2:
                resultat["nombre_canaux"] = 1
            elif pixels.ndim == 3:
                resultat["nombre_canaux"] = pixels.shape[2]

            resultat["ecart_type_pixels"] = float(pixels.std())

    except (UnidentifiedImageError, OSError, SyntaxError):
        resultat["corrompue"] = True

    return resultat

### 1.3 : Parcourir tout le dataset

Le dataset est parcouru classe par classe.

Chaque image est analysée individuellement et son résultat est ajouté à une
liste qui sera ensuite transformée en DataFrame Pandas.

In [ ]:
resultats = []

for classe in CLASSES:
    dossier_classe = RAW_DIR / classe

    for image_path in sorted(dossier_classe.iterdir()):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            resultat = analyser_image(image_path, classe)
            resultats.append(resultat)

df_images = pd.DataFrame(resultats)

df_images.head()

In [ ]:
#verif

print(f"Nombre total de fichiers analysés : {len(df_images)}")

df_images.info()

### 1.4 : Faire un premier résumé

Le DataFrame permet maintenant d'avoir une vue globale du dataset.

Nous vérifions notamment le nombre d'images analysées et la présence
éventuelle d'images corrompues.

In [ ]:
print("Nombre total d'images :", len(df_images))
print("Nombre d'images corrompues :", df_images["corrompue"].sum())
print("Nombre d'images valides :", (~df_images["corrompue"]).sum())

In [22]:
df_images["classe"].value_counts()

classe
paper        252
plastic      224
glass        188
cardboard    169
metal        149
trash         50
Name: count, dtype: int64

In [ ]:
df_images[
    [
        "nom",
        "classe",
        "format",
        "mode",
        "largeur",
        "hauteur",
        "ecart_type_pixels",
        "nombre_canaux",
        "taille_octets",
        "corrompue",
    ]
].head(10)

### 2.1 : Créer la fonction de détection d'images corrompues

Une image corrompue est une image qui ne peut pas être correctement ouverte
ou vérifiée par la bibliothèque d'analyse.

Nous créons une fonction dédiée qui reçoit le chemin d'une image et retourne
`True` si elle est corrompue, sinon `False`.

Cette fonction permet d'isoler la détection de l'intégrité des images et
pourra être réutilisée dans les étapes suivantes.

In [19]:
def est_image_corrompue(image_path):
    """Retourne True si l'image est corrompue, sinon False."""
    
    try:
        with Image.open(image_path) as image:
            image.verify()
        return False

    except (UnidentifiedImageError, OSError, SyntaxError):
        return True

### 2.2 : Tester la fonction sur le dataset

La fonction est appliquée à toutes les images du dossier `data/raw/`.

Les résultats sont ajoutés au DataFrame d'exploration afin d'identifier
précisément les fichiers corrompus.

In [21]:
df_images["chemin"] = df_images.apply(
    lambda ligne: RAW_DIR / ligne["classe"] / ligne["nom"],
    axis=1
)

df_images["corrompue"] = df_images["chemin"].apply(est_image_corrompue)

###  2.3 : Afficher les résultats

Nous comptons les images corrompues puis affichons leurs informations afin
de pouvoir les identifier précisément.

In [25]:
nombre_corrompues = df_images["corrompue"].sum()

print(f"Nombre d'images corrompues : {nombre_corrompues}")

Nombre d'images corrompues : 6


In [24]:
images_corrompues = df_images[df_images["corrompue"]]

images_corrompues[
    ["nom", "classe", "taille_octets", "corrompue"]
]

,nom,classe,taille_octets,corrompue
147,cardboard83.jpg,cardboard,20888,True
326,glass74.jpg,glass,7184,True
446,metal48.jpg,metal,7259,True
633,paper213.jpg,paper,7293,True
791,plastic13.jpg,plastic,5250,True
1004,trash3.jpg,trash,9192,True


### 2.4 : Vérification globale

In [26]:
print("===== AUDIT DES IMAGES CORROMPUES =====")
print(f"Images analysées   : {len(df_images)}")
print(f"Images corrompues  : {df_images['corrompue'].sum()}")
print(f"Images valides     : {(~df_images['corrompue']).sum()}")

===== AUDIT DES IMAGES CORROMPUES =====
Images analysées   : 1032
Images corrompues  : 6
Images valides     : 1026


## 3.1 Fonction de détection des images vides

Une image est considérée comme vide lorsqu'elle est :

- entièrement noire ;
- entièrement blanche ;
- ou présente une très faible variation entre ses pixels.

Une fonction dédiée permet d'identifier ces images sans modifier le dataset original.

In [29]:
def est_image_vide(image_path, seuil_variation=5):
    """Détecte une image noire, blanche ou présentant peu de variation."""

    try:
        with Image.open(image_path) as image:
            pixels = np.asarray(image.convert("RGB"))

        # image entièrement noire
        if np.all(pixels == 0):
            return True

        # image entièrement blanche
        if np.all(pixels == 255):
            return True

        # image présentant très peu de variation
        seuil_variation=5
        if pixels.std() <= seuil_variation:
            return True

        return False

    except (UnidentifiedImageError, OSError, SyntaxError):
        return False